# ML00 · 機器學習總覽

用圖與直覺建立機器學習 machine learning 的整體心智模型，先看懂「學習在學什麼」，再進入後續技術細節。

## 學習目標
- 能用自己的話說明機器學習 machine learning 是什麼，以及它與傳統寫死規則的差別。
- 能區分監督式 supervised 與非監督式 unsupervised 學習，並判斷一個問題屬於分類 classification 還是迴歸 regression。
- 理解特徵 feature 與標籤 label 的角色，知道資料如何餵給模型。
- 建立「學習就是最小化誤差」的直覺，看懂損失 loss 的意義。
- 理解為什麼要做訓練/驗證/測試切分 train/validation/test split，以及它如何防止自欺。
- 能用圖說明過擬合 overfitting 與泛化 generalization，知道「背答案」與「真會」的差別。

In [ ]:
# 中文字型設定（本書會畫圖才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 什麼是機器學習

機器學習 machine learning 是「讓程式從資料中自己找出規則」的方法，而不是由人把規則一條條寫死。

傳統的規則式程式 rule-based program 是人先想好判斷邏輯（例如「樓高超過 60 公尺就算高樓」），再寫成 if 條件。資料驅動 data-driven 的做法則相反：給一批帶答案的樣本，讓模型 model 自己抓出門檻，最後得到一個會做預測 prediction 的函數。

整體畫面就是一句話：**給資料，得到一個會預測的函數**。何時用機器學習？當規則太複雜、人難以窮舉，或規則會隨資料變動時。

In [ ]:
# 概念：規則式程式 vs 資料驅動，對比「人寫死門檻」與「模型自己抓門檻」
import numpy as np

# 造一批假樓高資料（公尺）與是否為高樓的標準答案當示範用
heights = np.array([18, 25, 33, 45, 58, 66, 72, 85, 95, 120])  # 每棟樓的高度
is_high = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])             # 1 表示高樓，0 表示非高樓

# 規則式：人事先寫死門檻 60 公尺
rule_threshold = 60
rule_pred = (heights > rule_threshold).astype(int)  # 人寫的規則直接判斷

# 資料驅動：不寫死門檻，從資料自己找「最後一棟非高樓」與「第一棟高樓」的中點
last_low = heights[is_high == 0].max()              # 標籤為 0 的最高樓
first_high = heights[is_high == 1].min()            # 標籤為 1 的最矮樓
learned_threshold = (last_low + first_high) / 2     # 模型從資料長出的門檻
learned_pred = (heights > learned_threshold).astype(int)

print('人寫死的門檻：', rule_threshold, '公尺')
print('從資料學到的門檻：', learned_threshold, '公尺')
print('學到的門檻與標準答案是否完全一致：', bool((learned_pred == is_high).all()))

## 學習的兩大家族：監督式與非監督式

有沒有標準答案（標籤 label）決定了學習屬於哪個家族。

監督式學習 supervised learning 有標籤，模型學「輸入到答案」的對應；非監督式學習 unsupervised learning 沒有標籤，目標是在資料裡找出結構。

監督式又依輸出型態分兩種：

| 任務 | 輸出型態 | 例子 |
|---|---|---|
| 迴歸 regression | 連續數值 | 預測房價 |
| 分類 classification | 類別標籤 | 判斷住宅 / 商辦 |
| 分群 clustering（非監督式） | 自動分組 | 把街區分群 |

In [ ]:
# 概念：用同一份小資料對照迴歸、分類、分群三種任務
# 造一張假街區資料：面積（平方公尺）、房價（萬元）、用途類別
area = np.array([60, 85, 120, 45, 200, 150])         # 特徵：面積
price = np.array([900, 1200, 1700, 700, 2600, 2100]) # 迴歸的標籤：連續房價
usage = np.array(['住宅', '住宅', '商辦', '住宅', '商辦', '商辦'])  # 分類的標籤：類別

# 迴歸：輸出是連續數值，這裡看房價的範圍
print('迴歸任務（預測房價）標籤範圍：', price.min(), '到', price.max(), '萬元')

# 分類：輸出是有限類別
print('分類任務（住宅/商辦）類別有：', set(usage))

# 分群：沒有用到任何標籤，只看特徵就把街區依面積分成兩組
# 技巧：最簡單的分群可用一個門檻把樣本切兩群，體會「無標籤找結構」
group = np.where(area < area.mean(), '小面積群', '大面積群')
for a, g in zip(area, group):
    print(f'面積 {a:>3} -> {g}')

## 資料的語言：特徵與標籤

模型只看得懂數字化的資料，因此要先學會用資料表的語言描述問題。

- 樣本 sample：一列就是一個觀測對象（例如一棟房屋）。
- 特徵 feature：拿來當輸入的那些欄（樓高、面積、屋齡）。
- 特徵向量 feature vector：一個樣本所有特徵組成的一排數字。
- 標籤 label：要預測的那一欄（房價）。

理解「一列是一個樣本、欄是特徵、要預測的那一欄是標籤」，是後續所有內容的基礎。

In [ ]:
# 概念：特徵矩陣 X 與標籤向量 y 的擺放方式
# 造一張含樓高、面積、屋齡的小表，最後一欄房價當標籤
feature_names = ['樓高(m)', '面積(m2)', '屋齡(年)']
X = np.array([
    [18,  60,  5],
    [33,  85, 12],
    [66, 120,  2],
    [95, 200, 20],
])                      # 每一列是一個樣本，每一欄是一個特徵
y = np.array([900, 1200, 1700, 2600])  # 標籤：房價（萬元），與 X 的列一一對應

print('特徵欄位：', feature_names)
print('特徵矩陣 X 形狀（樣本數, 特徵數）：', X.shape)
print('標籤向量 y 形狀（樣本數,）：', y.shape)
print('第 0 個樣本的特徵向量：', X[0], '對應標籤：', y[0])

## 學習就是最小化誤差：損失的直覺

學習的核心動作是「先猜，再看錯多少，再調整讓錯變小」。

損失 loss 是一個衡量「預測與真值差多少」的數字；誤差 error 越大，損失越大。學習就是不斷調整模型，讓損失最小化 minimization。

一個最常見的損失是均方誤差 mean squared error：把每個樣本的誤差平方後取平均。直覺上，一條直線越貼近資料點，總誤差越小，損失也越小。

In [ ]:
# 概念：用均方誤差比較幾條候選直線誰更貼近資料
# 造一組面積 -> 房價的假散點（帶一點雜訊）
rng = np.random.default_rng(0)                 # 固定亂數種子，結果可重現
area = np.linspace(50, 200, 20)                # 面積從 50 到 200 取 20 點
price = 10 * area + 200 + rng.normal(0, 80, 20)  # 真實關係加上雜訊當作觀測房價

def mse(slope, intercept):
    pred = slope * area + intercept            # 用直線做預測
    return np.mean((price - pred) ** 2)        # 均方誤差：誤差平方後取平均

# 三條候選直線（斜率, 截距），其中一條接近真實關係
candidates = [(6, 400), (10, 200), (14, 0)]
for slope, intercept in candidates:
    print(f'直線 y={slope}x+{intercept:<4} 的損失(MSE)= {mse(slope, intercept):.1f}')

plt.figure(figsize=(6, 4))
plt.scatter(area, price, color='gray', label='資料點')
for slope, intercept in candidates:
    plt.plot(area, slope * area + intercept, label=f'y={slope}x+{intercept}')
plt.xlabel('面積(m2)'); plt.ylabel('房價(萬元)')
plt.title('不同直線的擬合與損失')
plt.legend(); plt.show()

## 不要自欺：資料切分與評估

在看過的資料上表現好不算數，因為模型可能只是把答案背起來。

標準做法是把資料切成三份：

| 子集 | 用途 |
|---|---|
| 訓練集 training set | 拿來學、調整模型 |
| 驗證集 validation set | 調參數、選模型時偷看的成績 |
| 測試集 test set | 全程不碰，最後一次驗收 |

用沒看過的資料評估，才知道模型能不能泛化 generalization，避免「考古題全對、真考試掛掉」。

In [ ]:
# 概念：把資料隨機切成訓練/驗證/測試三份
n = 20
samples = np.arange(n)                          # 假設有 20 個樣本，用編號代表
rng = np.random.default_rng(42)
shuffled = rng.permutation(samples)             # 先打亂，避免原始順序帶來偏差

# 注意：切分比例常見為 70/15/15，依資料量調整
n_train = int(n * 0.7)
n_val = int(n * 0.15)
train_idx = shuffled[:n_train]                  # 前 70% 拿來訓練
val_idx = shuffled[n_train:n_train + n_val]     # 接著 15% 拿來驗證
test_idx = shuffled[n_train + n_val:]           # 剩下留作測試，全程不碰

print('訓練集樣本數：', len(train_idx))
print('驗證集樣本數：', len(val_idx))
print('測試集樣本數：', len(test_idx))
print('三份是否互不重疊：', len(set(train_idx) & set(test_idx)) == 0)

## 過擬合與泛化：背答案 vs 真會

把訓練資料背得太細，反而會把雜訊也學進去，換新資料就失準。

- 欠擬合 underfitting：模型太簡單，連訓練資料都學不好。
- 過擬合 overfitting：模型太彎，訓練誤差很小但測試誤差大。
- 泛化 generalization：訓練與測試表現都好，才是「真會」。

判斷關鍵是看模型複雜度 model complexity 與兩組誤差的落差：訓練好、測試差，就是背答案。

In [ ]:
# 概念：用不同次數的多項式擬合，對照欠擬合、適中、過擬合
rng = np.random.default_rng(1)
x = np.linspace(0, 1, 15)                        # 自造帶雜訊的點
y_true = np.sin(2 * np.pi * x)                   # 背後真實規律
y = y_true + rng.normal(0, 0.25, x.shape)        # 加雜訊得到觀測值

xs = np.linspace(0, 1, 200)                       # 畫平滑曲線用的密集點
degrees = {'欠擬合(次數1)': 1, '適中(次數3)': 3, '過擬合(次數12)': 12}

plt.figure(figsize=(6, 4))
plt.scatter(x, y, color='gray', label='資料點')
for label, deg in degrees.items():
    coef = np.polyfit(x, y, deg)                 # 用多項式擬合，次數越高越能彎曲
    train_err = np.mean((np.polyval(coef, x) - y) ** 2)
    plt.plot(xs, np.polyval(coef, xs), label=f'{label} 訓練誤差={train_err:.3f}')
plt.ylim(-2, 2)
plt.xlabel('x'); plt.ylabel('y')
plt.title('欠擬合 / 適中 / 過擬合')
plt.legend(fontsize=8); plt.show()

## 練習

以下三題由淺入深，皆為複合型但可快速完成。資料請自己用 numpy / list 造，不要引用任何外部檔案。

In [ ]:
# TODO 1 ·（簡單）幫房屋資料貼上正確標籤（整合：特徵與標籤 + 監督式分類與迴歸）
#   情境：你拿到一小批自造的房屋資料，每列有樓高、面積、屋齡，以及房價與「住宅/商辦」兩個欄位。
#   要求：
#     1. 用 numpy / list 自造這份小資料，並指出哪些欄是特徵 feature、哪一欄會被當成標籤 label。
#     2. 判斷「預測房價」與「判斷住宅或商辦」各屬於迴歸 regression 還是分類 classification，並印出說明依據。
#   驗收：應該看到一份清楚標示特徵與標籤的對照，以及兩個任務各自被正確歸類為迴歸或分類。
# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）用最小誤差挑一條房價線並誠實評估（整合：損失直覺 + 資料切分 + 監督式迴歸）
#   情境：用自造的「面積 -> 房價」散點，模擬一個最簡單的迴歸問題。
#   要求：
#     1. 先把資料切成訓練/驗證/測試 train/validation/test split。
#     2. 在訓練資料上比較兩三條候選直線，依損失 loss（誤差大小）選出較好的一條。
#     3. 用沒看過的測試資料計算被選中直線的損失，判斷是否仍然準。
#   驗收：應該看到被選中的直線在訓練集上誤差最小，並能說出它在測試集上是否一樣表現良好。
# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）找出背答案的那條曲線（整合：過擬合與泛化 + 損失 + 資料切分 + 模型複雜度）
#   情境：用自造、帶一點雜訊的容積率與樓高關係資料，準備三種複雜度不同的擬合線（很直、適中、很彎）。
#   要求：
#     1. 切出訓練集與測試集，對三種擬合線分別計算它們在兩組資料上的損失 loss。
#     2. 指出哪一條是過擬合 overfitting、哪一條欠擬合、哪一條泛化 generalization 最好，並說明從兩組誤差的落差如何判斷。
#     3. 簡短印出：若想避免過擬合，可以朝哪個方向調整（如降低複雜度或增加資料）。
#   驗收：應該看到「訓練誤差很小但測試誤差明顯變大」的那條被指認為過擬合，泛化最佳的那條在兩組資料上表現接近。
# TODO: 在這裡完成

## 小結

- 機器學習 machine learning 是從資料中長出規則，核心畫面是「給資料，得到一個會預測的函數」。
- 有沒有標籤 label 決定學習家族：監督式 supervised 分為迴歸 regression（連續數值）與分類 classification（類別），非監督式 unsupervised 則做分群 clustering 找結構。
- 資料用「列是樣本 sample、欄是特徵 feature、要預測的那欄是標籤 label」的語言描述。
- 學習就是最小化損失 loss：先猜、看誤差、再調整，讓預測越來越貼近真值。
- 用訓練/驗證/測試切分 train/validation/test split 在沒看過的資料上評估，才能避免自欺。
- 真會等於能泛化 generalization；訓練好但測試差就是過擬合 overfitting，背了答案卻沒學到規律。